In [ ]:
!pip install pinecone

In [ ]:
import os
import openai
import pinecone

In [ ]:
# Load API keys
with open(r"C:\mindful-ai\sapient-ds\2024\presentation\week-06\openai-api-key-purushotham.txt") as f:
    openai_api_key = f.read().strip()

In [ ]:
with open(r"C:\mindful-ai\sapient-ds\2024\presentation\week-06\pinecone-examples\key.txt") as f:
    pinecone_api_key = f.read().strip()
pinecone_env = "us-west-2"
pinecone_api_key

In [ ]:
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone
pc = Pinecone(api_key=pinecone_api_key)

# Index name
index_name = "medical-knowledge"

# Create index if not exists
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,        # embedding size for text-embedding-ada-002
        metric="cosine",       # similarity metric
        spec=ServerlessSpec(
            cloud="aws",       # "aws" or "gcp"
            region="us-east-1" # must be one of: aws us-east-1, aws us-west-2, gcp us-central1
        )
    )

# Connect to the index
index = pc.Index(index_name)
print("Connected to index:", index_name)


In [ ]:
docs = [
    {"id": "1", "text": "Metformin is commonly prescribed as a first-line treatment for type 2 diabetes."},
    {"id": "2", "text": "Insulin therapy may be required if lifestyle changes and oral medications are not sufficient."},
    {"id": "3", "text": "GLP-1 receptor agonists help improve blood sugar control and support weight loss."},
    {"id": "4", "text": "Type 1 diabetes requires lifelong insulin therapy as the pancreas produces no insulin."}
]


In [ ]:
from openai import OpenAI

# initialize client
client = OpenAI(api_key=openai_api_key)

# Generate embeddings
response = client.embeddings.create(
    model="text-embedding-ada-002",
    input=[doc["text"] for doc in docs]
)

# Prepare Pinecone vectors
vectors = [
    {
        "id": docs[i]["id"],
        "values": response.data[i].embedding,
        "metadata": {"text": docs[i]["text"]}
    }
    for i in range(len(docs))
]

# Upsert into Pinecone
index.upsert(vectors=vectors)

print("Inserted", len(vectors), "documents into Pinecone.")



In [ ]:
from openai import OpenAI

# Initialize client
client = OpenAI(api_key=openai_api_key)

# Your query
query = "What drugs are used to manage type 2 diabetes?"

# Generate embedding for the query using new API
query_response = client.embeddings.create(
    model="text-embedding-ada-002",
    input=query
)

query_embedding = query_response.data[0].embedding

# Query Pinecone
results = index.query(vector=query_embedding, top_k=3, include_metadata=True)

# Display results
for match in results["matches"]:
    print(f"Score: {match['score']:.4f}, Text: {match['metadata']['text']}")


